In [5]:
print("MM21B030")

MM21B030


# **Architecture_sweep**

In [ ]:
sweep_config = {
    'name': 'architecture_search',
    'method': 'grid',
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {'values': [5]},
        'embedding_dim': {'values': [64,128]},
        'hidden_dim': {'values': [64, 128, 256]},
        'cell_type': {'values': ['LSTM', 'GRU', 'RNN']},
        'num_layers_encoder': {'values': [1, 2]},
        'dropout': {'values': [0.2, 0.3]},
        'batch_size': {'values': [32]},
        'beam_size': {'values': [1]},
        'learning_rate': {'values' : [10e-3, 10e-4]}
        }
}

In [ ]:
# Run python vanilla/vanilla_sweep.py

# **Fine_search Sweep**

In [ ]:
sweep_config = {
    'name': 'fine_search',
    'method': 'grid',
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {'values': [10, 15]},
        'embedding_dim': {'values': [64]},
        'hidden_dim': {'values': [256]},
        'cell_type': {'values': ['LSTM']},
        'num_layers_encoder': {'values': [1]},
        'dropout': {'values': [0.2]},
        'batch_size': {'values': [32, 64]},
        'beam_size': {'values': [1, 2]},
        'learning_rate': {'values' : [5e-3, 10e-3, 15e-3]}
        }
}


In [ ]:
# Run python vanilla/vanilla_sweep.py

# **Testing Best Model**

In [16]:
import load_data 
import importlib

# Reload both modules

importlib.reload(load_data)

from seq2seq_model import Seq2SeqModel 
from load_data import get_data_loaders 
import torch
import torch.nn as nn

# Load data
train_loader, dev_loader, test_loader, char_to_idx_latin, char_to_idx_devanagari = get_data_loaders()

# Input is Latin, Output is Devanagari
input_vocab_size = len(char_to_idx_latin)
output_vocab_size = len(char_to_idx_devanagari)

# Index maps
idx_to_latin = {v: k for k, v in char_to_idx_latin.items()}
idx_to_devanagari = {v: k for k, v in char_to_idx_devanagari.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
model = Seq2SeqModel(
    input_vocab_size=input_vocab_size,
    output_vocab_size=output_vocab_size,
    embedding_dim=64,
    hidden_dim=256,
    cell_type="lstm",
    num_layers_encoder=1,
    num_layers_decoder=1,
    dropout=0.2,
    device=device,
    beam_size=2
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.005)

# Training loop
best_val_loss = float('inf')
for epoch in range(10):
    model.train()
    train_loss = 0
    
    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)

        output = model(src, trg[:, :-1])  # teacher forcing
        loss = criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, trg in dev_loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg[:, :-1])
            val_loss += criterion(
                output.reshape(-1, output_vocab_size),
                trg[:, 1:].reshape(-1)
            ).item()
    
    val_loss /= len(dev_loader)
    print(f'Epoch {epoch+1}: Train Loss = {train_loss/len(train_loader):.4f}, Val Loss = {val_loss:.4f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_vanilla_model.pt')
        print("Saved new best model")




Epoch 1: Train Loss = 1.5778, Val Loss = 0.8511
Saved new best model
Epoch 2: Train Loss = 0.7829, Val Loss = 0.6967
Saved new best model
Epoch 3: Train Loss = 0.6489, Val Loss = 0.6495
Saved new best model
Epoch 4: Train Loss = 0.5858, Val Loss = 0.6249
Saved new best model
Epoch 5: Train Loss = 0.5405, Val Loss = 0.6149
Saved new best model
Epoch 6: Train Loss = 0.5071, Val Loss = 0.6088
Saved new best model
Epoch 7: Train Loss = 0.4873, Val Loss = 0.6056
Saved new best model
Epoch 8: Train Loss = 0.4693, Val Loss = 0.5976
Saved new best model
Epoch 9: Train Loss = 0.4543, Val Loss = 0.5988
Epoch 10: Train Loss = 0.4448, Val Loss = 0.6091


In [19]:
# Load best model
model.load_state_dict(torch.load('best_vanilla_model.pt'))

# Helper: Decode sequence
def decode_sequence(indices, idx_to_char):
    return ''.join([idx_to_char.get(idx, '') for idx in indices if idx not in {0, 1, 2}])

# Testing loop
model.eval()
test_loss = 0
exact_matches = 0
total_sequences = 0
all_predictions = []  # Store ALL predictions

with torch.no_grad():
    for src, trg in test_loader:
        src, trg = src.to(device), trg.to(device)
        output = model(src, trg[:, :-1])

        # Loss calculation
        test_loss += criterion(
            output.reshape(-1, output_vocab_size),
            trg[:, 1:].reshape(-1)
        ).item()
        
        # Get predictions
        _, predicted = output.max(2)
        batch_size = predicted.size(0)
        
        for i in range(batch_size):
            input_seq = decode_sequence(src[i].cpu().numpy(), idx_to_latin)
            pred_seq = decode_sequence(predicted[i].cpu().numpy(), idx_to_devanagari)
            true_seq = decode_sequence(trg[i, 1:].cpu().numpy(), idx_to_devanagari)
            
            if pred_seq == true_seq:
                exact_matches += 1
            total_sequences += 1
            
            all_predictions.append((input_seq, pred_seq, true_seq))

# Calculate metrics
test_loss /= len(test_loader)
accuracy = exact_matches / total_sequences

# Print summary
print(f'\nTest Loss: {test_loss:.4f}, Exact Match Accuracy: {accuracy:.2%}')
print(f'Total Test Examples: {total_sequences}')

# Save ALL predictions to TSV (overwrites existing file)
with open("predictions_vanilla.tsv", "w", encoding="utf-8") as f:
    f.write("latin\tpredicted\tground_truth\n")
    for input_seq, pred_seq, true_seq in all_predictions:
        f.write(f"{input_seq}\t{pred_seq}\t{true_seq}\n")

print("\nFirst 10 Example Predictions:")
for i, (input_seq, pred_seq, true_seq) in enumerate(all_predictions[:10]):
    status = "✓" if pred_seq == true_seq else "✗"
    print(f"{i+1}. Input: {input_seq}")
    print(f"   Pred: {pred_seq}")
    print(f"   True: {true_seq} ({status})\n")

print(f"Saved ALL {total_sequences} predictions to predictions_vanilla.tsv")


Test Loss: 0.5967, Exact Match Accuracy: 29.21%
Total Test Examples: 4502

First 10 Example Predictions:
1. Input: ank
   Pred: अंक
   True: अंक (✓)

2. Input: anka
   Pred: अंकाा
   True: अंक (✗)

3. Input: ankit
   Pred: अंकित
   True: अंकित (✓)

4. Input: anakon
   Pred: अनकों
   True: अंकों (✗)

5. Input: ankhon
   Pred: अंकों
   True: अंकों (✓)

6. Input: ankon
   Pred: अंकों
   True: अंकों (✓)

7. Input: angkor
   Pred: अंगरर
   True: अंकोर (✗)

8. Input: ankor
   Pred: अंकरर
   True: अंकोर (✗)

9. Input: angaarak
   Pred: अंगररक
   True: अंगारक (✗)

10. Input: angarak
   Pred: अंगररक
   True: अंगारक (✗)

Saved ALL 4502 predictions to predictions_vanilla.tsv
